# Lab: an AI assistant that screens scam messages
### Thailand Cyber & Digital Innovation Challenge — Online Workshop
**Claude on Microsoft Foundry (Azure) · English edition**

---

### What you get from this notebook
An AI assistant that reads Thai messages and says whether each is **scam / normal / misinformation**, with its reasons,
and pulls out what matters: links, phone numbers, account numbers.

### Three stages
| Stage | Time | What you do | Who |
|---|---|---|---|
| **1** | 15 min | Run what is provided and see the first results | **Everyone** |
| **2** | 15 min | Change the prompt, add your own messages | **Everyone** |
| **3** | 15 min | Code track → tool use · design track → design the flow | Pick one |

> ### ⚠️ About the key
> The key you get is shared by your team, **has a spending cap, and is revoked this evening**.
> Never put it in code you push to GitHub, and never share it outside your team.

> ### Opening this file
> On the repository page, click **Code**, then **Codespaces → Create codespace on main**.
> VS Code opens in your browser with Python ready; nothing to install.
> If VS Code asks *Do you trust the authors of the files in this folder?*, click **Trust Folder & Continue**
> (Cancel leaves you in Restricted Mode, where cells cannot run).
> Open this file: top right should already show **Python 3.11**. If it still says *Select Kernel*, click
> **Select Kernel → Python Environments... → Python 3.11** (`/usr/local/bin/python`), never *Create Python Environment*.
> Then run the cells one by one.

---
## Stage 1 — make it run

In [ ]:
# Run this cell first (already installed in your team's Codespace, so it finishes almost at once)
!pip install -q anthropic

### Add your team key
Replace the two values below with the ones you were given, then run the cell.

In [ ]:
FOUNDRY_API_KEY  = "paste-your-team-key-here"
FOUNDRY_RESOURCE = "paste-your-resource-name-here"

from anthropic import AnthropicFoundry

client = AnthropicFoundry(api_key=FOUNDRY_API_KEY, resource=FOUNDRY_RESOURCE)

# The two models we use. Their prices differ by about 40 times
MODEL_FAST  = "claude-haiku-5-5"   # cheap and fast: reads every message
MODEL_SMART = "claude-opus-5-5"    # smarter and pricier: only for cases that need thought

# One test call
r = client.messages.create(
    model=MODEL_FAST,
    max_tokens=200,
    messages=[{"role": "user", "content": "Reply briefly: ready"}],
)
print("".join(b.text for b in r.content if b.type == "text"))
print("\n✅ Connected to Azure. You can move on.")

**If you see ✅, you are through the hardest step of the whole lab.**

Stuck here? Paste the whole error line in the Teams chat; the instructor answers at each checkpoint.

---
### The sample messages
Every message below was **written for teaching**: all phone numbers, links and account numbers are fake.
The messages are in Thai, as the tool is built for Thai text; the comment on each line says what it says.

The second value on each line is the **answer key**, used to measure accuracy later.

In [ ]:
SAMPLE_MESSAGES = [
    ("พัสดุของท่านตกค้างที่ศุลกากร กรุณาชำระค่าธรรมเนียม 128 บาท ภายใน 24 ชม. ที่ th-parcel-pay.example", "scam"),  # parcel held at customs: pay a THB 128 fee at a link within 24 h
    ("คุณได้รับสิทธิ์เงินกู้ฉุกเฉิน 50,000 บาท ดอกเบี้ย 0% ไม่เช็คเครดิต สมัครเลย add LINE: @fastloan-x", "scam"),  # THB 50,000 emergency loan, 0% interest, no credit check: add a chat ID
    ("ธนาคารแจ้ง บัญชีของท่านถูกระงับชั่วคราว ยืนยันตัวตนภายในวันนี้ที่ bank-verify-th.example เพื่อปลดล็อก", "scam"),  # "your bank": account suspended, verify today at a link to unlock it
    ("ที่นี่กองปราบปราม บัญชีคุณพัวพันคดีฟอกเงิน ติดต่อกลับด่วน 02-000-0000 มิฉะนั้นจะออกหมายจับ", "scam"),  # "police": your account is tied to money laundering, call back or face arrest
    ("รับสมัครงานพาร์ทไทม์ กดไลก์คลิปละ 50 บาท โอนไว ทักแชทรับงานเลย", "scam"),  # part-time job: THB 50 per video you like, message us
    ("กลุ่มลงทุนทองคำ การันตีกำไร 15% ต่อเดือน รับประกันทุน รีบสมัคร เหลือ 3 ที่", "scam"),  # gold investment: guaranteed 15% a month, capital protected, 3 places left
    ("การไฟฟ้าแจ้งค้างชำระ 1,240 บาท ตัดไฟคืนนี้ ชำระที่ power-bill-th.example ภายใน 2 ชม.", "scam"),  # "electricity authority": THB 1,240 overdue, power cut tonight, pay at a link
    ("แม่จ๊ะ นี่ลูกนะ เปลี่ยนเบอร์ใหม่ พอดีมีเรื่องด่วน โอนเงินให้หน่อย 8,000 บัญชี XXX-X-XXXXX-X", "scam"),  # "Mum, it's me, new number": urgent, transfer THB 8,000
    ("ยินดีด้วย! คุณคือผู้โชคดีได้รับโทรศัพท์รุ่นล่าสุด กรอกข้อมูลรับของที่ win-prize-th.example", "scam"),  # you won the latest phone: fill in your details at a link
    ("แอปกู้เงินด่วน อนุมัติใน 5 นาที ไม่ต้องค้ำ โหลดได้ที่ลิงก์นี้ ไม่ต้องผ่านสโตร์", "scam"),  # instant loan app, approved in 5 minutes, install outside the store

    ("รหัส OTP ของคุณคือ 884213 ใช้ได้ 5 นาที ธนาคารไม่มีนโยบายขอรหัสนี้จากคุณ", "normal"),  # your OTP; the bank never asks for it
    ("บัญชี xxx-123 มีเงินเข้า 2,500.00 บาท คงเหลือ 18,430.25 บาท เวลา 14:02", "normal"),  # account credited THB 2,500.00
    ("แจ้งเตือน นัดหมายพบแพทย์ พรุ่งนี้ 10:30 น. แผนกอายุรกรรม กรุณามาก่อนเวลา 15 นาที", "normal"),  # doctor's appointment reminder
    ("พัสดุเลขที่ TH00123456 ถึงจุดคัดแยกแล้ว คาดว่าจัดส่งพรุ่งนี้ ตรวจสอบสถานะในแอป", "normal"),  # parcel reached the sorting centre, delivery tomorrow
    ("โปรโมชันสมาชิก ลด 20% เมื่อซื้อครบ 500 บาท ที่ร้านสาขาใดก็ได้ ถึงสิ้นเดือนนี้", "normal"),  # members get 20% off over THB 500 until month end
    ("มหาวิทยาลัยแจ้ง เปิดลงทะเบียนเรียนภาคปลาย 1-15 ธ.ค. ผ่านระบบทะเบียนของมหาวิทยาลัย", "normal"),  # university: course registration 1-15 Dec

    ("ด่วน! น้ำมะนาวผสมโซดาอุ่น ฆ่าเซลล์มะเร็งได้ 100% หมอไม่อยากให้คุณรู้ แชร์ต่อด่วน", "misinformation"),  # warm lemon soda kills 100% of cancer cells, doctors hide it, share now
    ("เขื่อนแตกแล้ว! น้ำจะถึงตัวเมืองใน 2 ชม. ทางการปิดข่าว รีบอพยพ ส่งต่อให้คนที่คุณรัก", "misinformation"),  # the dam has burst, officials are hiding it, forward this
    ("ยืนยันแล้ว รัฐจะเก็บภาษีเงินฝากทุกบัญชี 3% เริ่มเดือนหน้า รีบถอนก่อนโดน", "misinformation"),  # "confirmed": a 3% tax on every deposit next month, withdraw now
    ("คลิปนี้พิสูจน์ว่าคนดังพูดแบบนี้จริง ดูแล้วอึ้ง แชร์ก่อนโดนลบ", "misinformation"),  # this clip proves a celebrity said it, share before it is deleted
]

print(f"{len(SAMPLE_MESSAGES)} messages in total")
for text, answer in SAMPLE_MESSAGES[:3]:
    print(f"  [{answer}] {text[:50]}...")

### Stage 1 triage — the cheap model (Haiku)

Two things make this code usable for real, not just a toy.

**`format`** forces the answer into the JSON shape we define, not free text,
so a program can use it straight away with no text parsing.

**`effort: low`** tells the model this task needs little thought, saving money and time.

In [ ]:
import json

TRIAGE_SCHEMA = {
    "type": "json_schema",
    "schema": {
        "type": "object",
        "properties": {
            "category": {"type": "string", "enum": ["scam", "normal", "misinformation"]},
            "confidence": {"type": "number"},
            "warning_signals": {"type": "array", "items": {"type": "string"}},
        },
        "required": ["category", "confidence", "warning_signals"],
        "additionalProperties": False,
    },
}

TRIAGE_PROMPT = """You screen messages for scams. Most messages are in Thai.
Read the message and put it in exactly one of three categories.

- scam = tries to get money or personal data, or to make someone open a dangerous link
- normal = an ordinary message with no intent to deceive
- misinformation = false or distorted information designed to be shared

confidence is a number from 0 to 1.
warning_signals lists what made you decide, as short English phrases."""


def triage(text):
    """Send one message to the triage stage and return the result as a dict."""
    r = client.messages.create(
        model=MODEL_FAST,
        max_tokens=2000,
        system=TRIAGE_PROMPT,
        messages=[{"role": "user", "content": text}],
        output_config={"effort": "low", "format": TRIAGE_SCHEMA},
    )
    out = json.loads(next(b.text for b in r.content if b.type == "text"))
    out["_tokens"] = (r.usage.input_tokens, r.usage.output_tokens)
    return out


# Try the first message
result = triage(SAMPLE_MESSAGES[0][0])
print(json.dumps(result, ensure_ascii=False, indent=2))

### Run the whole set

A note for anyone who has used the Anthropic API before: Foundry has **no Message Batches**,
so we loop over the messages one at a time.

In [ ]:
results = []
for text, answer in SAMPLE_MESSAGES:
    out = triage(text)
    results.append((text, answer, out))
    mark = "✅" if out["category"] == answer else "❌"
    print(f'{mark} said={out["category"]:<15} answer={answer:<15} {text[:38]}...')

n_correct = sum(1 for _, a, o in results if o["category"] == a)
print(f"\nAccuracy {n_correct}/{len(results)} = {n_correct / len(results) * 100:.0f}%")

---
## Stage 2 — make it yours

This is the heart of the session: **a well-written prompt matters more than a more expensive model.**

Try one change at a time, re-run the `results` cell above, and watch whether the accuracy moves.

1. Add a rule to `TRIAGE_PROMPT`, such as *"Treat any pressure to decide within a time limit as a warning signal."*
2. Add a category to the `enum`, such as `"suspicious"` for unclear cases
3. Paste scam messages your team has actually received below

In [ ]:
my_messages = [
    "Paste a message you have really received here",
    "You can add several",
]

for text in my_messages:
    out = triage(text)
    print(f'{out["category"]:<15} (confidence {out["confidence"]:.2f})  {text[:45]}')
    print(f'   signals: {", ".join(out["warning_signals"])}\n')

---
## Stage 3 (code track) — a second stage with the smart model

Stage 1 says "probably a scam", but the person who got the message wants to know **what now?**

Stage 2 uses Opus to explain why, and to pull out what can actually be reported or blocked.
It only runs on cases stage 1 marked as scam — **not on every message**.
This is where the biggest saving comes from.

In [ ]:
DETAIL_SCHEMA = {
    "type": "json_schema",
    "schema": {
        "type": "object",
        "properties": {
            "explanation": {"type": "string"},
            "links": {"type": "array", "items": {"type": "string"}},
            "phone_numbers": {"type": "array", "items": {"type": "string"}},
            "account_numbers": {"type": "array", "items": {"type": "string"}},
            "next_steps": {"type": "array", "items": {"type": "string"}},
        },
        "required": ["explanation", "links", "phone_numbers", "account_numbers", "next_steps"],
        "additionalProperties": False,
    },
}

DETAIL_PROMPT = (
    "You are a cybercrime specialist. "
    "Explain in plain English a non-technical victim can follow, with no jargon. "
    "If a kind of item is not present, return an empty list for it."
)


def analyze_detail(text):
    r = client.messages.create(
        model=MODEL_SMART,
        max_tokens=4000,
        system=DETAIL_PROMPT,
        messages=[{"role": "user", "content": text}],
        output_config={"effort": "medium", "format": DETAIL_SCHEMA},
    )
    return json.loads(next(b.text for b in r.content if b.type == "text"))


scam_cases = [t for t, _, o in results if o["category"] == "scam"]
print(f"Stage 1 passed on {len(scam_cases)} of {len(results)} messages "
      f"— {len(results) - len(scam_cases)} expensive calls avoided\n")

detail = analyze_detail(scam_cases[0])
print(json.dumps(detail, ensure_ascii=False, indent=2))

### Going further: let the model call a tool

So far the model judges from the text alone. A real system also checks against outside data.

`strict: True` guarantees the model always sends arguments in exactly the shape we define,
so nothing odd reaches the program and breaks it.

> **Important** On Claude Opus 5.5 you **cannot force a tool call**
> (a forcing `tool_choice` errors). The right way is to say clearly in the prompt when the tool should be used.

In [ ]:
# A made-up blocklist for teaching. In a real system this is an agency API or a shared blocklist
BLOCKLIST = {"th-parcel-pay.example", "bank-verify-th.example", "power-bill-th.example"}

TOOLS = [{
    "name": "check_domain",
    "description": "Check whether a domain is on the blocklist. Use it whenever the message contains a link.",
    "strict": True,
    "input_schema": {
        "type": "object",
        "properties": {"domain": {"type": "string"}},
        "required": ["domain"],
        "additionalProperties": False,
    },
}]

test_message = SAMPLE_MESSAGES[0][0]

r = client.messages.create(
    model=MODEL_SMART,
    max_tokens=4000,
    system="If the message contains a link, always call the check_domain tool before answering.",
    messages=[{"role": "user", "content": test_message}],
    tools=TOOLS,
)

for b in r.content:
    if b.type == "tool_use":
        domain = b.input["domain"]
        listed = domain in BLOCKLIST
        print(f"Model asked to check: {domain}  →  {'🚨 on the blocklist' if listed else 'not on the blocklist'}")
    elif b.type == "text" and b.text.strip():
        print(b.text)

print("\n(Next step your team can take: send the tool_result back and let the model sum up)")

---
## Stage 3 (design track — no code)

You can do the most important part of a proposal without touching code:
**designing how this reaches the people who need it.**

Open the JSON results from stages 1–2 and answer these four questions in your team document.

1. **Who is the user?** Older people? Students? Bank staff? Pick one group.
2. **When do they meet the tool?** Before clicking a link? While about to pay? After being scammed?
3. **Through which channel?** A chat bot? An app? A browser extension? A number to forward messages to?
4. **What do they see on screen?** Sketch one screen, and place `category`, `confidence` and `warning_signals` on it.

> Hackathon winners are rarely the teams with the most accurate model.
> They are the teams with the clearest answers to these four questions.

---
## Measure how accurate it is

The first question any judge asks is **"how do you know it works?"**
Teams that cannot answer lose marks here every time.

The answer is not a feeling. It is a test set you built yourself.

In [ ]:
wrong = [(t, a, o["category"]) for t, a, o in results if o["category"] != a]
n_correct = len(results) - len(wrong)

print(f"Overall accuracy: {n_correct}/{len(results)} = {n_correct / len(results) * 100:.0f}%\n")

print("Accuracy per category")
for label in ["scam", "normal", "misinformation"]:
    group = [(a, o) for _, a, o in results if a == label]
    hit = sum(1 for a, o in group if o["category"] == a)
    print(f"  {label:<15} {hit}/{len(group)}")

print("\nMisses — read every one; they are what you tune the prompt with")
if not wrong:
    print("  (none this round — find harder messages to add)")
for t, a, pred in wrong:
    print(f"  answer {a} but said {pred}: {t[:55]}...")

print("\n--- A caution to write into your proposal ---")
print("Calling a scam 'normal' is far more dangerous than calling a normal message 'scam':")
print("the first lets the victim get scammed; the second is only annoying.")
print("When you design, decide which way you will err, and explain why to the judges.")

### What it really costs

This figure belongs in your proposal,
because **a solution too expensive to run is a solution nobody gets to use.**

In [ ]:
PRICE = {  # US dollars per million tokens
    MODEL_FAST:  {"in": 0.10, "out": 0.50},
    MODEL_SMART: {"in": 4.00, "out": 20.00},
}
THB_PER_USD = 33  # set to the rate on the day you use it

tok_in  = sum(o["_tokens"][0] for _, _, o in results)
tok_out = sum(o["_tokens"][1] for _, _, o in results)

cost_usd = (tok_in / 1e6) * PRICE[MODEL_FAST]["in"] + (tok_out / 1e6) * PRICE[MODEL_FAST]["out"]
per_msg_usd = cost_usd / len(results)

print(f"Triage of {len(results)} messages used {tok_in:,} input / {tok_out:,} output tokens")
print(f"That is USD {per_msg_usd:.6f} (THB {per_msg_usd * THB_PER_USD:.4f}) per message\n")
print(f"10,000 messages a month                     = USD {per_msg_usd * 10000:,.2f} (THB {per_msg_usd * 10000 * THB_PER_USD:,.0f})")
print(f"With the expensive model on everything (~40x) = USD {per_msg_usd * 10000 * 40:,.2f} (THB {per_msg_usd * 10000 * 40 * THB_PER_USD:,.0f})")
print("\nThat is why we split it into two stages")

---
## A real case: an e-mail that asks for nothing

The messages so far were easy scams to spot, because they ask for money or a click.

The more dangerous ones **ask for nothing that looks suspicious**. The example below is an e-mail
a company really received (names and numbers removed; only the structure is left).
It is in Thai; an English translation follows the code.

Read it first and decide what you think it is, then run the next cell.

*Translation: "Hello [your company] team. [Sender] from [agency] here. We are a marketing agency working with technology brands,
looking for partners in Thailand for AI, workstation and server activities. Do you build workstations to customer
specifications, including storage upgrades from other brands? Could you introduce the person who handles workstations
or business partnerships? If convenient, could you share that person's LINE or WhatsApp? We would like to introduce
ourselves and discuss details. Thank you."*

In [ ]:
business_email = """สวัสดีค่ะ ทีมงาน [บริษัทของคุณ]

[ชื่อผู้ส่ง] จาก [ชื่อเอเจนซี] นะคะ เราเป็นเอเจนซีการตลาดที่ทำงานกับแบรนด์เทคโนโลยี
และกำลังมองหาพาร์ตเนอร์ในไทยเพื่อร่วมทำกิจกรรมด้าน AI, Workstation และ Server ค่ะ

อยากสอบถามว่า [บริษัทของคุณ] รับจัด Workstation ตามความต้องการของลูกค้า
รวมถึงอัปเกรดอุปกรณ์จัดเก็บข้อมูลจากแบรนด์อื่นด้วยไหมคะ
และพอจะแนะนำผู้ที่ดูแลเรื่อง Workstation หรือการร่วมมือทางธุรกิจให้พูดคุยด้วยได้หรือเปล่าคะ

ถ้าสะดวก รบกวนขอ LINE หรือ WhatsApp ของผู้ดูแลด้วยนะคะ
ทางเราอยากติดต่อแนะนำตัวและพูดคุยรายละเอียดเพิ่มเติมค่ะ

ขอบคุณมากค่ะ"""

verdict = triage(business_email)
print("Stage 1 says:", verdict["category"], f"(confidence {verdict['confidence']:.2f})")
print("Signals seen:", ", ".join(verdict["warning_signals"]) or "none")

### Why stage 1 cannot catch it

This e-mail has no link, no attachment, asks for no code, sets no deadline and asks for no money; the language is polite and correct.
**Judging from the text alone will never be enough**, because what is wrong sits outside the text.

What it really asks for is **the name and private contact of the person who can approve purchases** — the first step
towards impersonating a supplier later, not a scam inside this e-mail.

Check outside information. The easiest question to answer: **does this agency even have a website?**

In [ ]:
import socket

def has_website(domain):
    """Check whether a domain points at a real server. Works with no extra library."""
    try:
        socket.gethostbyname(domain)
        return True
    except socket.gaierror:
        return False


# A domain with a real site, compared with one without
for d in ["github.com", "example-agency-with-no-site.example"]:
    print(f"{d:45s} {'has a website' if has_website(d) else 'no website'}")

print()
print("--- What the real case showed ---")
print("The sender's domain was registered 7 years ago but has no website at all")
print("The mail is hosted in a different country from the one claimed")
print("No DMARC, so the domain is easy to spoof")
print("The mailbox is a per-country campaign name, not a person")
print()
print("A marketing agency with no website of its own contradicts the very business it claims")

### Lessons to take away

| | Scams in the dataset | The real case above |
|---|---|---|
| What it asks for | Money, or a click | A person's name and private contact |
| Can the text decide it? | Yes | No: check outside information |
| When the harm happens | At once | Later, when a supplier is impersonated |

**If your team builds a tool for organisations**, do not make it answer only scam or not-scam.
Give it a **suspicious** state too, saying what to check next,
because the costliest cases usually look the most normal.

---
## Next question: who wrote this message?

The old way to spot scams was spelling mistakes and odd language. **That no longer works**,
because criminals use AI to translate and polish their writing better than many real people.

So the more useful question is not "is the language good?" but **"is it good in the way people in this trade actually write?"**

Let the model assess the author of the same e-mail.

In [ ]:
AUTHOR_SCHEMA = {
    "type": "json_schema",
    "schema": {
        "type": "object",
        "properties": {
            "author": {
                "type": "string",
                "enum": ["native Thai writer", "non-native writer", "machine-translated or AI-written"],
            },
            "confidence": {"type": "number"},
            "language_signals": {"type": "array", "items": {"type": "string"}},
            "unnatural_phrases": {"type": "array", "items": {"type": "string"}},
        },
        "required": ["author", "confidence", "language_signals", "unnatural_phrases"],
        "additionalProperties": False,
    },
}

AUTHOR_PROMPT = """You are an expert in written Thai. Read the message and judge who most likely wrote it.

Look for
- sentence structures translated directly from English
- dictionary translations instead of the words people in that trade actually use
- whether the polite particles (ka, khrap, na ka) are well placed, and unnaturally uniform
- the complete absence of any first-person pronoun
- Thai business phrases that should be there but are missing
- English-style punctuation

Write language_signals in English. Quote unnatural_phrases exactly as they appear in the message."""


def analyze_author(text):
    r = client.messages.create(
        model=MODEL_SMART,
        max_tokens=4000,
        system=AUTHOR_PROMPT,
        messages=[{"role": "user", "content": text}],
        output_config={"effort": "medium", "format": AUTHOR_SCHEMA},
    )
    return json.loads(next(b.text for b in r.content if b.type == "text"))


author = analyze_author(business_email)
print("Author:", author["author"], f"(confidence {author['confidence']:.2f})")
print()
print("Language signals")
for s in author["language_signals"]:
    print("  -", s)
print()
print("Unnatural phrases")
for s in author["unnatural_phrases"]:
    print("  -", s)

### Signals that work on Thai text

| What you see | Why it is suspicious |
|---|---|
| Dictionary terms such as อุปกรณ์จัดเก็บข้อมูล ("data storage device") | People in the trade say สตอเรจ (storage), hard disk or SSD |
| A name followed by จาก ("from") with no pronoun | The English "Name from Company" pattern, carried over whole |
| No ดิฉัน or ผม ("I") anywhere | Thai business e-mails use a first-person pronoun at least once |
| Polite particles all correct, yet perfectly uniform | Real writers vary more than that |
| Not a single typo | Real sales e-mails usually slip somewhere |
| No Thai business phrases such as เบื้องต้น, ทั้งนี้ | A translated template has no local idiom |
| English-style commas separating list items | Thai separates with a plain space |

### The most important caution in this section

**AI-written does not mean fraud.** Many companies now use AI to translate e-mails to overseas customers as a matter of course.
If your tool calls something a scam because AI wrote it, you will accuse a great many innocent people.

Treat this result as **one signal among many**. This case is suspicious because three things come together:
language not written by a Thai speaker, a domain with no website, and a request for the purchasing manager's private contact.
The first alone proves nothing.

---
## From prototype to real use — mask personal data before logging

Your tool takes in **other people's messages**, which often carry phone numbers, account numbers or names.

Keep raw logs and, the day the system is breached, what leaks is the data of people who trusted you.
The right way is to **mask before you log, not log first and delete later.**

Run the cell below, then extend it to cover the patterns your team actually sees.

In [ ]:
import re

def mask_pii(text):
    """Mask personal data before anything is logged or passed on."""
    text = re.sub(r"\b0\d{1,2}[-\s]?\d{3}[-\s]?\d{3,4}\b", "[PHONE]", text)
    text = re.sub(r"\b\d{3}[-\s]?\d[-\s]?\d{5}[-\s]?\d\b", "[ACCOUNT]", text)
    text = re.sub(r"\bhttps?://\S+|\b[\w.-]+\.(?:com|net|example)\S*", "[LINK]", text)
    return text


# Every sample, plus one made-up line with an account number (fake data, not in the test set)
examples = [t for t, _ in SAMPLE_MESSAGES] + ["โอนค่ามัดจำเข้าบัญชี 123-4-56789-0 แล้วส่งสลิปมาที่ 081-234-5678"]

for text in examples:
    after = mask_pii(text)
    if after != text:
        print("before:", text)
        print("masked:", after, "\n")

print("--- Decisions your team must make, and write in your team document ---")
print("1. How long do you keep raw messages, and why?")
print("2. Who on the team can read the logs?")
print("3. If a user asks you to delete their data, can your system do it?")

---
## After today

### For teams in the Thailand Cyber & Digital Innovation Challenge — Concept Proposal due 31 December 2026

What to take from this notebook into your proposal

- The problem you chose and who the user is (the four questions in stage 3)
- An architecture picture: message in → cheap triage → detailed analysis → result to the user
- **An accuracy figure from your own test set**, saying plainly which cases still fail
- The cost per message, and for 10,000 messages a month
- A scope you can finish in a three-day hackathon — **small and working beats big and unfinished**

### A last reminder
The keys handed out today are revoked this evening.
If you want to keep going, ask the organisers about model access for the proposal stage.

---
*Thailand Cyber & Digital Innovation Challenge · U.S. Embassy Bangkok · depa*